# MusicBrainz Data Pipeline —  Extracting & Transforming Data

## 1. Import Libraries and Configure MusicBrainz API Connection

### Purpose

* Prepare the Python environment and define the MusicBrainz API connection settings used by the extraction script.

* This section imports the required libraries for API requests, data transformation, rate-limit control, and timestamp generation. It also defines the MusicBrainz base API URL and a custom `User-Agent` header so the API can identify the application making requests.

In [1]:
import os
import time
from pathlib import Path
from datetime import datetime, timezone

import requests
import pandas as pd
from dotenv import load_dotenv

In [2]:
# Load environment variables from .env
load_dotenv()

BASE_URL = "https://musicbrainz.org/ws/2"
EMAIL = os.getenv("USER_AGENT_EMAIL")

if EMAIL is None:
    raise ValueError("USER_AGENT_EMAIL is not set. Please add it to your .env file.")

HEADERS = {
    "User-Agent": f"musicbrainz-etl-project/1.0 ({EMAIL})"
}

## 2. Define Target Artists for Extraction

### Purpose

Create a list of artists that will be used as search inputs for the MusicBrainz API. Each artist name will be passed into a recording search request to retrieve related songs, releases, and artist metadata.

In [3]:
target_artists = [
    "Coldplay",
    "Taylor Swift",
    "Dua Lipa",
    "James Blunt",
    "BTS"
]


## 3. Extract Raw Recording Data from MusicBrainz API

### Purpose:

* Send search requests to the MusicBrainz recording endpoint for each target artist and collect the raw nested JSON responses.

* Each response is stored with the original search keyword and extraction timestamp. This preserves the raw API data before transformation, which supports data lineage and allows the transformation step to be re-run later if needed.

In [4]:
raw_data = []

for artist in target_artists:
    params = {
        "query": f'artist:"{artist}"',
        "limit": 50,
        "fmt": "json"
    }
    
    response = requests.get(
        f"{BASE_URL}/recording",
        headers=HEADERS,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    raw_data.append({
        "artist_search": artist,
        "extracted_at": datetime.utcnow().isoformat(),
        "data": data
    })

    time.sleep(1.1)

print("Extraction completed")
print(f"Number of artist searches completed: {len(raw_data)}")


Extraction completed
Number of artist searches completed: 5


## 4. Understanding the MusicBrainz Data Structure


### Purpose

* The raw response from MusicBrainz is nested JSON. Before transforming the data, we inspect one recording object to identify where song, artist, and album/release fields are stored.

* This step helps us map the raw JSON structure into three clean tables: songs, artists, and albums.

In [5]:
first_recording = raw_data[0]["data"]["recordings"][0]
first_artist = first_recording["artist-credit"][0]["artist"]
first_release = first_recording["releases"][0]

print("----------------------SONG--------------------------------------------")
print(first_recording["id"])
print(first_recording["title"])
print(first_recording["length"])
print(first_recording.get("first-release-date"))

print("----------------------ARTIST-------------------------------------------")
print(first_artist["id"])
print(first_artist["name"])

print("----------------------ALBUM--------------------------------------------")
print(first_release["id"])
print(first_release["title"])
print(first_release.get("date"))

----------------------SONG--------------------------------------------
a1b972f8-0ee5-40f0-94dd-82f645071caf
Trouble
261960
2001-05-14
----------------------ARTIST-------------------------------------------
cc197bad-dc9c-440d-a5b5-d52ba2e14234
Coldplay
----------------------ALBUM--------------------------------------------
6cae3863-54c0-4bbb-bf56-c4f9ee450ea7
Clubbed Out
2001-05-14


## 5. Extract Album / Release Data

### Purpose

* Extract album/release information from the nested MusicBrainz raw JSON response and store it in a clean album table. 

* In MusicBrainz, album-like information is stored inside the `releases` list of each recording.

In [6]:
album_list = []

for row in raw_data:
    artist_search = row["artist_search"]
    extracted_at = row["extracted_at"]

    recordings = row["data"].get("recordings", [])

    for recording in recordings:
        releases = recording.get("releases", [])

        for release in releases:
            album_id = release.get("id")

            album_list.append({
                "album_id": album_id,
                "album_name": release.get("title"),
                "release_date": release.get("date"),
                "country": release.get("country"),
                "status": release.get("status"),
                "track_count": release.get("track-count"),
                "album_url": f"https://musicbrainz.org/release/{album_id}" if album_id else None,
                "artist_search": artist_search,
                "extracted_at": extracted_at
            })

album_df = pd.DataFrame(album_list)

album_df.head()

,album_id,album_name,release_date,country,status,track_count,album_url,artist_search,extracted_at
0,6cae3863-54c0-4bbb-bf56-c4f9ee450ea7,Clubbed Out,2001-05-14,GB,Official,35,https://musicbrainz.org/release/6cae3863-54c0-...,Coldplay,2026-07-07T08:10:45.138346
1,f0017216-fb89-4002-a9c5-311486d7dc66,"2002-06-26: Wisseloord Studios, Hilversum, The...",None,None,Promotion,7,https://musicbrainz.org/release/f0017216-fb89-...,Coldplay,2026-07-07T08:10:45.138346
2,f0017216-fb89-4002-a9c5-311486d7dc66,"2002-06-26: Wisseloord Studios, Hilversum, The...",None,None,Promotion,7,https://musicbrainz.org/release/f0017216-fb89-...,Coldplay,2026-07-07T08:10:45.138346
3,d1bbb3b5-5ca7-4039-99c6-789e97b80b6c,A Rush of B-Sides to the Head,None,None,Bootleg,21,https://musicbrainz.org/release/d1bbb3b5-5ca7-...,Coldplay,2026-07-07T08:10:45.138346
4,157c3d31-3977-45fb-a70f-1b4eed779345,Yellow,2003-02-28,XW,Official,3,https://musicbrainz.org/release/157c3d31-3977-...,Coldplay,2026-07-07T08:10:45.138346


In [7]:
album_df.shape

(1268, 9)

## 6. Extract Artist Data

### Purpose:

* Parse artist-level information from the nested MusicBrainz raw JSON response and store it in a clean artist table. 
* In MusicBrainz, artist information is stored inside the `artist-credit` list of each recording.

In [8]:
artist_list = []

for row in raw_data:
    artist_search = row["artist_search"]
    extracted_at = row["extracted_at"]

    recordings = row["data"].get("recordings", [])

    for recording in recordings:
        artist_credits = recording.get("artist-credit", [])

        for artist_info in artist_credits:
            artist = artist_info.get("artist", {})
            artist_id = artist.get("id")

            artist_list.append({
                "artist_id": artist_id,
                "artist_name": artist.get("name"),
                "artist_sort_name": artist.get("sort-name"),
                "artist_type": artist.get("type"),
                "artist_country": artist.get("country"),
                "artist_disambiguation": artist.get("disambiguation"),
                "artist_url": f"https://musicbrainz.org/artist/{artist_id}" if artist_id else None,
                "artist_search": artist_search,
                "extracted_at": extracted_at
            })

artist_df = pd.DataFrame(artist_list)

artist_df.head()

,artist_id,artist_name,artist_sort_name,artist_type,artist_country,artist_disambiguation,artist_url,artist_search,extracted_at
0,cc197bad-dc9c-440d-a5b5-d52ba2e14234,Coldplay,Coldplay,None,None,None,https://musicbrainz.org/artist/cc197bad-dc9c-4...,Coldplay,2026-07-07T08:10:45.138346
1,cc197bad-dc9c-440d-a5b5-d52ba2e14234,Coldplay,Coldplay,None,None,None,https://musicbrainz.org/artist/cc197bad-dc9c-4...,Coldplay,2026-07-07T08:10:45.138346
2,cc197bad-dc9c-440d-a5b5-d52ba2e14234,Coldplay,Coldplay,None,None,None,https://musicbrainz.org/artist/cc197bad-dc9c-4...,Coldplay,2026-07-07T08:10:45.138346
3,cc197bad-dc9c-440d-a5b5-d52ba2e14234,Coldplay,Coldplay,None,None,None,https://musicbrainz.org/artist/cc197bad-dc9c-4...,Coldplay,2026-07-07T08:10:45.138346
4,cc197bad-dc9c-440d-a5b5-d52ba2e14234,Coldplay,Coldplay,None,None,None,https://musicbrainz.org/artist/cc197bad-dc9c-4...,Coldplay,2026-07-07T08:10:45.138346


In [9]:
artist_df.shape

(250, 9)

## 7. Extract Song / Recording Data

### Purpose:

* Parse recording-level information from the nested MusicBrainz raw JSON response and store it in a clean song table. 
* In MusicBrainz, each item inside the `recordings` list represents one song-level recording.

In [10]:
song_list = []

for row in raw_data:
    artist_search = row["artist_search"]
    extracted_at  = row["extracted_at"]

    recordings = row["data"].get("recordings", [])
    
    for recording in recordings:
       song_list.append({
        "recording_id": recording.get("id"),
        "title": recording.get("title"),
        "length_ms": recording.get("length"),
        "video": recording.get("video"),
        "score": recording.get("score"),
        "first_release_date": recording.get("first-release-date"),
        "artist_search": artist_search,
        "extracted_at": extracted_at
       })

song_df = pd.DataFrame(song_list)

song_df.head()

In [11]:
song_df.shape

(250, 13)

## 8. Clean and Validate Transformed Data

### Purpose:

Remove duplicate records, convert date columns into proper datetime format, handle simple missing values, and validate the final transformed tables before saving them.

In [12]:
# Remove duplicate records
album_df = album_df.drop_duplicates(subset=["album_id"]).reset_index(drop=True)
artist_df = artist_df.drop_duplicates(subset=["artist_id"]).reset_index(drop=True)
song_df = song_df.drop_duplicates(subset=["recording_id"]).reset_index(drop=True)

print("Album shape after removing duplicates:", album_df.shape)
print("Artist shape after removing duplicates:", artist_df.shape)
print("Song shape after removing duplicates:", song_df.shape)

In [13]:
print("Album missing values:")
print(album_df.isna().sum())

print("\nArtist missing values:")
print(artist_df.isna().sum())

print("\nSong missing values:")
print(song_df.isna().sum())

Album missing values:
album_id           0
album_name         0
release_date     126
country          126
status             8
track_count        0
album_url          0
artist_search      0
extracted_at       0
dtype: int64

Artist missing values:
artist_id                0
artist_name              0
artist_sort_name         0
artist_type              5
artist_country           5
artist_disambiguation    4
artist_url               0
artist_search            0
extracted_at             0
dtype: int64

Song missing values:
song_id                 0
song_name               0
duration_ms            30
first_release_date     43
score                   0
is_video              226
song_url                0
artist_id               0
artist_name             0
album_id                5
album_name              5
artist_search           0
extracted_at            0
dtype: int64


In [14]:
# Convert date columns
album_df["release_date"] = pd.to_datetime(album_df["release_date"], errors="coerce")
album_df["extracted_at"] = pd.to_datetime(album_df["extracted_at"], errors="coerce")

artist_df["extracted_at"] = pd.to_datetime(artist_df["extracted_at"], errors="coerce")

song_df["first_release_date"] = pd.to_datetime(song_df["first_release_date"], errors="coerce")
song_df["extracted_at"] = pd.to_datetime(song_df["extracted_at"], errors="coerce")

In [15]:
# Fill missing video flag
song_df["video"] = song_df["video"].fillna(False).astype(bool)

In [16]:
print("Album missing values:")
print(album_df.isna().sum())

print("\nArtist missing values:")
print(artist_df.isna().sum())

print("\nSong missing values:")
print(song_df.isna().sum())

Album missing values:
album_id           0
album_name         0
release_date     245
country          126
status             8
track_count        0
album_url          0
artist_search      0
extracted_at       0
dtype: int64

Artist missing values:
artist_id                0
artist_name              0
artist_sort_name         0
artist_type              5
artist_country           5
artist_disambiguation    4
artist_url               0
artist_search            0
extracted_at             0
dtype: int64

Song missing values:
song_id                0
song_name              0
duration_ms           30
first_release_date    89
score                  0
is_video               0
song_url               0
artist_id              0
artist_name            0
album_id               5
album_name             5
artist_search          0
extracted_at           0
dtype: int64


In [17]:
print("Album data types:")
print(album_df.dtypes)

print("\nArtist data types:")
print(artist_df.dtypes)

print("\nSong data types:")
print(song_df.dtypes)

Album data types:
album_id                 object
album_name               object
release_date     datetime64[ns]
country                  object
status                   object
track_count               int64
album_url                object
artist_search            object
extracted_at     datetime64[ns]
dtype: object

Artist data types:
artist_id                        object
artist_name                      object
artist_sort_name                 object
artist_type                      object
artist_country                   object
artist_disambiguation            object
artist_url                       object
artist_search                    object
extracted_at             datetime64[ns]
dtype: object

Song data types:
song_id                       object
song_name                     object
duration_ms                  float64
first_release_date    datetime64[ns]
score                          int64
is_video                        bool
song_url                      object
artist_id

In [18]:
print("Final album_df shape:", album_df.shape)
print("Final artist_df shape:", artist_df.shape)
print("Final song_df shape:", song_df.shape)

Final album_df shape: (639, 9)
Final artist_df shape: (5, 9)
Final song_df shape: (250, 13)


## 9. Save Transformed Data Locally

* Export the cleaned DataFrames as CSV files before moving the workflow to AWS S3.

* This local save step simulates what will later happen in AWS Lambda, where transformed files will be saved into the transformed S3 bucket.

In [19]:
output_path = Path("data/transformed")
output_path.mkdir(parents=True, exist_ok=True)

album_df.to_csv(output_path / "albums.csv", index=False)
artist_df.to_csv(output_path / "artists.csv", index=False)
song_df.to_csv(output_path / "songs.csv", index=False)

print("Transformed data saved locally.")

In [20]:
list(output_path.iterdir())

[WindowsPath('data/transformed/albums.csv'),
 WindowsPath('data/transformed/artists.csv'),
 WindowsPath('data/transformed/tracks.csv')]